# 📑 Hoist - CoCStatus
 
## 🔗 Links
- **[BRD - CoC Status Report - tbd](tbd)**  
  Comprehensive documentation outlining the business requirements for this report.
 
- **[Story Ticket - tbd](tbd)**  
  Details about the story ticket, including tasks and progress.
 
- **[UAT Findings Ticket - tbd](tbd)**  
  A list of findings from UAT.
 
---

#### Initialization

In [0]:
# %pip install openpyxl
# %restart_python

In [0]:
import os
import sys

dbutils.widgets.text("run_date", "")
dbutils.widgets.text("adhoc_run", '')

try:
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    repo_root = "/Workspace" + notebook_path.rsplit("/HoistInvestorReports/", 1)[0] 
    sys.path.insert(0, os.path.join(repo_root, "aws_utils"))
    sys.path.insert(0, os.path.join(repo_root, "01-utils"))
except Exception:
    sys.path.append("../../aws_utils")
    sys.path.append("../../01-utils")

from aws_utils import get_env_prefix, get_env_fullname
from helpers import load_configs, build_portfolio_filter, get_last_business_date_of_previous_month

env = get_env_prefix()


In [0]:
from datetime import datetime
from pytz import timezone
from pyspark.sql import functions as F
from pyspark.sql.window import Window

athens_tz = timezone('Europe/Athens')
adhoc_run = dbutils.widgets.get("adhoc_run")

if adhoc_run == '1':
    dbutils.widgets.text("run_date", '')
    run_date = dbutils.widgets.get("run_date")
else:
    run_date = spark.sql(f"SELECT MAX(BusinessDate) FROM `{env}-cdh`.primaryactive.exposure").collect()[0][0].strftime("%Y-%m-%d")
run_date_str = run_date.replace('-', '')
now_datetime_str = datetime.now(athens_tz).strftime('%Y%m%dT%H%M%S')


In [0]:
from typing import Any
from delta import DeltaTable  # type: ignore
from pyspark.sql.functions import (
    coalesce,
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    trim,
    when,
    months_between,
    dateadd,
    datediff,
    regexp_replace
)

from typing import Optional
import delta  # type: ignore
from pyspark.sql import SparkSession
import pandas as pd

In [0]:
## Portfolios In Scope - All hoist portfolios
PortfolioName = "('Cell', 'Orbit', 'Pearl Portfolio A', 'Mercury', 'Light')"    

# MigrationDate

In [0]:
migrationDate = spark.sql(f"""
                          SELECT min(cast(AssignmentDate as date))  AS migrationDate,
                          CASE WHEN portfolioName = 'Mercury' THEN '2019-01-01'
                               WHEN portfolioName = 'Orbit' THEN '2022-03-28'
                               WHEN portfolioName = 'Light' THEN '2022-11-07'
                               WHEN PortfolioName = 'Cell' THEN '2023-10-23' -- 2022-11-07
                               WHEN PortfolioName = 'Pearl Portfolio A' THEN '2024-02-19' -- 2022-11-07
                               ELSE MIN(CAST(AssignmentDate AS DATE)) END AS cutoffdate,
                          PortfolioName, 
                          Prefix
                          FROM `{env}-cdh`.primary.portfolio
                          WHERE PortfolioName IN {PortfolioName}
                          GROUP BY PortfolioName, Prefix
                      """)

migrationDate.createOrReplaceTempView("migrationDate")

# InitializationExposureInfo 

In [0]:
initializationexposureinfo = spark.sql(f"""                      
select Exposurekey, InitialAssignmentBalance, 
CASE WHEN LEFT(Exposurekey,3) = 'MRC' THEN 'Mercury' 
WHEN LEFT(Exposurekey,3) = 'ORB' THEN 'Orbit' 
WHEN LEFT(Exposurekey,3) = 'CEL' THEN 'Cell' 
WHEN LEFT(Exposurekey,3) = 'PRA' THEN 'Pearl Portfolio A' 
WHEN LEFT(Exposurekey,3) = 'LGT' THEN 'Light' 
ELSE NULL END as PortfolioName
from `{env}-cdh`.reference.initializationexposureinfo
"""
)

initializationexposureinfo.createOrReplaceTempView("initializationexposureinfo")

# CompletedCoC 

In [0]:
completedcoc = spark.sql(
f'''
select PortfolioName, AgreementKey      
from `{env}-ptools`.perimeters.cocthirdpartieslegal_opt c 
where c.PortfolioName IN {PortfolioName}
AND BusinessDate = (
    SELECT MAX(BusinessDate) FROM `{env}-ptools`.perimeters.cocthirdpartieslegal_opt WHERE PortfolioName IN {PortfolioName} AND BusinessDate <= '{run_date}'
)
AND CompletedCoCContractCalc = TRUE 
'''
)
completedcoc.createOrReplaceTempView("completedcoc")

# Reassignments 

In [0]:
reassignments = spark.sql(f"""                             
        select distinct re.LoanID
        from `{env}-reportinputfiles`.investor.hoist_reassignments as re
          """)
reassignments.createOrReplaceTempView("reassignments")

# exposure

In [0]:
exposure = spark.sql(f"""
SELECT DISTINCT de.PortfolioName, 
SourceSystem,
CASE WHEN LEFT(de.AgreementKey,3) = md.Prefix THEN de.AgreementKey ELSE CONCAT(md.Prefix,de.AgreementKey) END as AgreementKey,
CASE WHEN LEFT(de.ExposureKey,3) = md.Prefix THEN de.ExposureKey ELSE CONCAT(md.Prefix,de.ExposureKey) END as ExposureKey, 
CDI, 
COALESCE(IEX.InitialAssignmentBalance, de.OriginationAmount) as InitialAssignmentAmount,
ExposureStatus,
FirstLetterDate,
SecondLetterDate,
ThirdLetterDate, 
JustificationCode,
RANK() OVER (PARTITION BY de.ExposureKey, de.CDI ORDER BY de.BusinessDate DESC,
       CASE WHEN FirstLetterDate IS NULL THEN 0 ELSE 1 END DESC, 
        CASE WHEN SecondLetterDate IS NULL THEN 0 ELSE 1 END DESC, 
        CASE WHEN ThirdLetterDate IS NULL THEN 0 ELSE 1 END DESC) AS rnk
FROM `{env}-cdh`.primary.exposure de
LEFT JOIN MigrationDate md ON md.PortfolioName = de.PortfolioName
LEFT JOIN initializationexposureinfo IEX ON regexp_replace(iex.ExposureKey, '[^0-9]', '') = regexp_replace(de.ExposureKey, '[^0-9]', '') AND de.PortfolioName = IEX.PortfolioName
WHERE 1=1
AND sourcesystem in ('qc', 'qca') 
AND de.PortfolioName IN {PortfolioName}
AND NOT (FirstLetterDate IS NULL AND SecondLetterDate IS NULL AND ThirdLetterDate IS NULL)
and BusinessDate <= '{run_date}'
and IsActiveFlag = TRUE 
""")

exposure.createOrReplaceTempView("exposure")

# Interaction

In [0]:
interaction = spark.sql(
f'''
SELECT Exposurekey, 
CDI,
CAST(InteractionDate AS DATE) as InteractionDate, 
InteractionType, 
InteractionTypeKey,
InteractionTypeCode, 
InteractionCategory,
PortfolioName,
SourceSystem,
RANK() OVER (PARTITION BY Exposurekey ORDER BY InteractionDate DESC) AS rnk
FROM `{env}-cdh`.primary.get_interactions_by_business_date( '{run_date}' )
WHERE PortfolioName IN {PortfolioName}
and InteractionDate <= '{run_date}'
and (InteractionTypeCode IN ('3110','3111','3112','3005','3045','3073','3007', '3541', '3543') AND SourceSystem = 'qc'
    or SourceSystem = 'qca' AND InteractionTypeKey IN ('8899', '8725', '8701', '8703'))

'''
)
interaction.createOrReplaceTempView("interaction")

# Final Table

In [0]:
finaltbl0 = spark.sql(
f'''
SELECT 
e.PortfolioName, 
CASE WHEN c.AgreementKey IS NOT NULL THEN 1 ELSE 0 END AS CompletedCoCFlag,
e.* 
, RANK() OVER (PARTITION BY ExposureKey, CDI   ORDER BY ThirdLetter DESC, SecondLetter DESC, FirstLetter DESC) as rnk
FROM (
    SELECT 
    PortfolioName, 
    AgreementKey,
    ExposureKey, 
    CDI, 
    InitialAssignmentAmount,
    SUM(ifnull(FirstLetter,0)) as FirstLetter,
    SUM(ifnull(SecondLetter,0)) as SecondLetter,
    SUM(ifnull(ThirdLetter,0)) as ThirdLetter,
    SUM(ifnull(SolutionOffered,0)) as SolutionOffered,
    SUM(ifnull(CounterProposal,0)) as CounterProposal,
    SUM(ifnull(CounterProposalAccepted,0)) as CounterProposalAccepted,
    SUM(ifnull(CounterProposalRefused,0)) as CounterProposalRefused,
    MAX(IFNULL(NonCooperativeFlag,0)) as NonCooperativeFlag
    FROM (
    SELECT 
    de.ExposureKey 
    ,de.CDI 
    ,de.InitialAssignmentAmount 
    , de.PortfolioName 
    , de.AgreementKey

    , CASE WHEN de.FirstLetterDate > IFNULL(de.SecondLetterDate, '1111-01-01') 
        AND de.FirstLetterDate > IFNULL(de.ThirdLetterDate, '1111-01-01') 
        AND de.FirstLetterDate > md.CutOffDate THEN 1 END AS FirstLetter,

    CASE WHEN de.SecondLetterDate > IFNULL(de.FirstLetterDate, '1111-01-01') 
        AND de.SecondLetterDate > IFNULL(de.ThirdLetterDate, '1111-01-01') 
        AND de.SecondLetterDate > md.CutOffDate THEN 1 END AS SecondLetter,
        
    CASE WHEN de.ThirdLetterDate > IFNULL(de.FirstLetterDate, '1111-01-01') 
        AND de.ThirdLetterDate > IFNULL(de.SecondLetterDate, '1111-01-01') 
        AND de.ThirdLetterDate > md.CutOffDate THEN 1 END AS ThirdLetter,

    CASE WHEN de.ThirdLetterDate > IFNULL(de.FirstLetterDate, '1111-01-01') 
    AND de.ThirdLetterDate > IFNULL(de.SecondLetterDate, '1111-01-01') 
    AND de.ThirdLetterDate > md.CutOffDate 
    AND DATEDIFF(CAST('{run_date}' AS DATE), de.ThirdLetterDate) > 30
    AND de.ExposureStatus <> 'Writen Off' 
    THEN 1 
    END AS NonCooperativeFlag,
        
    CASE WHEN i.InteractionDate >= de.FirstLetterDate AND (InteractionTypeKey IN ('8899', '8725', '8701', '8703')  AND i.SourceSystem = 'qc'
            or i.SourceSystem = 'qca' AND InteractionTypeCode IN ('3110','3111','3112','3005', '3541', '3543')) THEN 1 END AS SolutionOffered,
    CASE WHEN i.InteractionDate >= de.FirstLetterDate AND (InteractionTypeKey IN ('8899', '8725')  AND i.SourceSystem = 'qc'
            or i.SourceSystem = 'qca' AND  InteractionTypeCode IN ('3045', '3073')) THEN 1 END AS CounterProposal,
    CASE WHEN i.InteractionDate >= de.FirstLetterDate AND (InteractionTypeKey = '8701'  AND i.SourceSystem = 'qc'
            or i.SourceSystem = 'qca' AND InteractionTypeCode IN ('3005', '3541')) THEN 1 END AS CounterProposalAccepted,
    CASE WHEN i.InteractionDate >= de.FirstLetterDate AND (InteractionTypeKey = '8703'  AND i.SourceSystem = 'qc'
            or i.SourceSystem = 'qca' AND InteractionTypeCode IN ('3007', '3543') )  THEN 1 END AS CounterProposalRefused 
    FROM exposure de 
    LEFT JOIN Interaction i
    on regexp_replace(de.ExposureKey, '[^0-9]', '') = regexp_replace(i.ExposureKey, '[^0-9]', '') and de.PortfolioName = i.PortfolioName and regexp_replace(de.CDI, '[^0-9]', '') = regexp_replace(i.CDI, '[^0-9]', '')
    LEFT JOIN MigrationDate md on de.PortfolioName = md.PortfolioName 
    LEFT JOIN reassignments as re on re.LoanID = de.ExposureKey
    WHERE de.rnk = 1
    AND RE.LoanID IS NULL AND (de.JUSTIFICATIONCODE IS NULL OR de.JUSTIFICATIONCODE NOT IN (43055,43056))
    ) 
    WHERE NOT (FirstLetter IS NULL AND SecondLetter IS NULL AND ThirdLetter IS NULL)
    GROUP BY AgreementKey, ExposureKey, CDI, InitialAssignmentAmount, PortfolioName
) e
LEFT JOIN CompletedCoC c 
ON regexp_replace(e.Agreementkey, '[^0-9]', '') = regexp_replace(c.Agreementkey, '[^0-9]', '') and e.PortfolioName = c.PortfolioName 

'''
)
finaltbl0.createOrReplaceTempView("finaltbl0")


In [0]:
Final = spark.sql(
f'''
SELECT 
cast('{run_date}' as date) as BusinessDate,
AgreementKey,
PortfolioName,
ExposureKey, 
CDI, 
cast(InitialAssignmentAmount as decimal(19,2)) as InitialAssignmentAmount,
case when CompletedCoCFlag = 1 then true else false end as CompletedCoCFlag, 
case when NonCooperativeFlag = 1 then true else false end as NonCooperativeFlag,
CASE WHEN FirstLetter >= 1 THEN true ELSE false END AS FirstLetterFlag,
CASE WHEN SecondLetter >= 1 THEN true ELSE false END AS SecondLetterFlag,
CASE WHEN ThirdLetter >= 1 THEN true ELSE false END AS ThirdLetterFlag,
CASE WHEN SolutionOffered >= 1 THEN true ELSE false END AS SolutionOfferedFlag,
CASE WHEN CounterProposal >= 1 THEN true ELSE false END AS CounterProposalFlag,
CASE WHEN CounterProposalAccepted >= 1 THEN true ELSE false END AS CounterProposalAcceptedFlag,
CASE WHEN CounterProposalRefused >= 1 THEN true ELSE false END as CounterProposalRefusedFlag
from finaltbl0
where rnk = 1

'''
)
Final.createOrReplaceTempView("Final")


#### Append Results

In [0]:
target_table_path_CoCStatus = f"`{env}-cdh-delivery`.investorreports.hoist_CoCStatus"

spark.sql(f"""DELETE FROM {target_table_path_CoCStatus} WHERE BusinessDate = '{run_date}' AND PortfolioName IN {PortfolioName}""")

Final = Final.withColumn("IngestionTimestamp", lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")))

Final.write \
        .format("delta") \
        .partitionBy("BusinessDate") \
        .mode("append") \
        .saveAsTable(target_table_path_CoCStatus) # in order to write in delta_lake


spark.sql(f"REPAIR TABLE {target_table_path_CoCStatus} SYNC METADATA")